In [ ]:
import os
import numpy as np
import pandas as pd

### sampling

In [ ]:
os.makedirs(f"../../../result/HEK293T_HCT116_sample/intersect50/input", exist_ok=True)

In [ ]:
embs = {}
for mod in ["BioPlex", "SECMX"]:
    for ctx in ["HCT116", "HEK293T"]:
        name = f"{mod}-{ctx}"
        embs[name] = pd.read_csv(
            f"../../../result/HEK293T_HCT116/input/{name}.tsv",
            sep="\t", index_col=0,
        )

In [ ]:
shared = sorted(
    set.intersection(*[set(df.index) for df in embs.values()])
)

In [ ]:
rng = np.random.default_rng(1)
n_keep = round(0.5 * len(shared))
drop = set(rng.permutation(shared)[n_keep:])

sampled = {}
for name, df in embs.items():
    sampled[name] = df[~df.index.isin(drop)]

In [ ]:
for name, df in sampled.items():
    df.to_csv(
        f"../../../result/HEK293T_HCT116_sample/intersect50/input/{name}.tsv",
        sep="\t", header=True, index=True,
    )

### Stats for future filtering

In [ ]:
# load genes in each experiment
ap_hct = set(sampled["BioPlex-HCT116"].index)
ap_hek = set(sampled["BioPlex-HEK293T"].index)
sec_hct = set(sampled["SECMX-HCT116"].index)
sec_hek = set(sampled["SECMX-HEK293T"].index)

In [ ]:
# total possible pairs in each cell line
hct_genes = ap_hct | sec_hct
hct_pairs = len(hct_genes) * (len(hct_genes)-1) // 2

hek_genes = ap_hek | sec_hek
hek_pairs = len(hek_genes) * (len(hek_genes)-1) // 2

In [ ]:
# top 1%, add other when needed
stat = pd.DataFrame(
    [
        [int(hct_pairs*0.01), int(hek_pairs*0.01)]
    ],
    index=["Percent1"],
    columns=["HCT116", "HEK293T"]
)
stat.to_csv("../../../result/HEK293T_HCT116_sample/intersect50/input/pair_stat.tsv", sep="\t")